# Churn Prediction Pipeline v2 (Modeling & Evaluation)

Notebook này chứa toàn bộ quy trình từ tiền xử lý, gán nhãn Churn v2, huấn luyện mô hình LightGBM, hiệu chuẩn xác suất và đánh giá mô hình sản xuất v2.

## 1. Khởi tạo Thư viện và Cấu hình

Chúng ta import các thư viện đặc trưng thời gian và thư viện học máy.

In [ ]:
import os
import datetime
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score

from src import config
from src.features.build_temporal_base import build_temporal_base_features
from src.features.build_lag_features import build_lag_features
from src.features.build_rolling_features import build_rolling_features
from src.features.build_trend_features import build_trend_features
from src.features.build_recency_features import build_recency_features
from src.training.evaluate import evaluate_predictions
from src.training.calibrate_model import fit_platt_calibration

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("ModelingNotebook")

## 2. Logic Gán Nhãn Churn v2

Khai báo các hàm gán nhãn Churn v2.

In [ ]:
import logging
import pandas as pd
import numpy as np
from src.data.load_silver import load_silver_table

logger = logging.getLogger("ModelingNotebook")

def datetime_ns(values: pd.Series) -> pd.Series:
    return pd.to_datetime(values, errors="coerce").astype("datetime64[ns]")


def next_event_after_snapshots(grid: pd.DataFrame, events: pd.DataFrame, date_column: str) -> pd.Series:
    left = grid[["customer_id", "snapshot_date"]].sort_values(["snapshot_date", "customer_id"])
    right = events[["customer_id", date_column]].dropna().sort_values([date_column, "customer_id"])
    left["snapshot_date"] = datetime_ns(left["snapshot_date"])
    right[date_column] = datetime_ns(right[date_column])
    joined = pd.merge_asof(
        left, right, left_on="snapshot_date", right_on=date_column,
        by="customer_id", direction="forward", allow_exact_matches=True,
    )
    return pd.Series(joined[date_column].to_numpy(), index=left.index).reindex(grid.index)


def build_churn_labels_v2(base: pd.DataFrame) -> pd.DataFrame:
    logger.info("Computing Churn Labels v2...")
    customers = load_silver_table("churn_customers")
    orders = load_silver_table("churn_orders")
    usage = load_silver_table("churn_product_usage")
    payments = load_silver_table("churn_payments")
    subscriptions = load_silver_table("churn_subscriptions")
    
    # Cast customer_id to int64 for all tables to avoid merge type issues
    base = base.copy()
    for df in (orders, usage, payments, subscriptions, customers, base):
        if "customer_id" in df.columns:
            df["customer_id"] = df["customer_id"].astype("int64")
            
    order_date = next(c for c in ("order_date", "created_at") if c in orders.columns)
    usage_date = next(c for c in ("event_date", "usage_date", "created_at") if c in usage.columns)
    payment_date = next(c for c in ("payment_date", "created_at") if c in payments.columns)
    subscription_date = next(c for c in ("start_date", "change_date", "created_at") if c in subscriptions.columns)
    
    orders[order_date] = pd.to_datetime(orders[order_date])
    usage[usage_date] = pd.to_datetime(usage[usage_date])
    payments[payment_date] = pd.to_datetime(payments[payment_date])
    subscriptions[subscription_date] = pd.to_datetime(subscriptions[subscription_date])
    customers["closed_date"] = pd.to_datetime(customers["closed_date"])
    
    # timezone naïve normalization
    for df in (orders, usage, payments, subscriptions, customers, base):
        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                if getattr(df[col].dt, "tz", None) is not None:
                    df[col] = df[col].dt.tz_localize(None)
                    
    order_status = next(c for c in ("status", "order_status") if c in orders.columns)
    payment_status = next(c for c in ("status", "payment_status") if c in payments.columns)
    orders["_is_completed"] = orders[order_status].astype(str).str.strip().str.lower().eq("completed").astype("int8")
    payments["_is_success"] = payments[payment_status].astype(str).str.strip().str.lower().eq("success").astype("int8")
    
    # Standardize and clean subscription plan tiers
    subscriptions["plan_tier_clean"] = subscriptions["plan_tier"].astype(str).str.strip()
    def clean_tier(val):
        v = str(val).strip().lower()
        if "free" in v:
            return "Free"
        elif "premium" in v:
            return "Premium"
        elif "plus" in v:
            return "Plus"
        return val
    subscriptions["plan_tier_clean"] = subscriptions["plan_tier_clean"].apply(clean_tier)
    
    # Sort subscriptions chronologically
    subscriptions = subscriptions.sort_values(["customer_id", subscription_date, "subscription_id"])
    
    # 1. Determine tier_at_snapshot
    grid_sub = base[["customer_id", "snapshot_date"]].copy()
    grid_sub["snapshot_date"] = datetime_ns(grid_sub["snapshot_date"])
    
    sub_lookup = subscriptions[["customer_id", subscription_date, "plan_tier_clean"]].copy()
    sub_lookup[subscription_date] = datetime_ns(sub_lookup[subscription_date])
    sub_lookup = sub_lookup.sort_values([subscription_date, "customer_id"])
    
    joined_tier = pd.merge_asof(
        grid_sub.sort_values(["snapshot_date", "customer_id"]),
        sub_lookup,
        left_on="snapshot_date",
        right_on=subscription_date,
        by="customer_id",
        direction="backward"
    )
    
    joined_tier = joined_tier.set_index(["customer_id", "snapshot_date"])
    base_indexed = base.set_index(["customer_id", "snapshot_date"])
    base["tier_at_snapshot"] = base_indexed.index.map(joined_tier["plan_tier_clean"])
    
    # 2. Determine downgrade to Free in next 30 days
    subscriptions["change_type_clean"] = subscriptions["change_type"].fillna("").astype(str).str.strip().str.lower()
    downgrade_to_free_events = subscriptions[
        (subscriptions["change_type_clean"] == "downgrade") & 
        (subscriptions["plan_tier_clean"] == "Free")
    ][["customer_id", subscription_date]].rename(columns={subscription_date: "downgrade_date"})
    
    next_downgrade_to_free = next_event_after_snapshots(base, downgrade_to_free_events, "downgrade_date")
    next_downgrade_to_free = pd.to_datetime(next_downgrade_to_free)
    
    horizon = base["snapshot_date"] + pd.Timedelta(days=30)
    downgrade_to_free_in_next_30d = (
        next_downgrade_to_free.notna() & 
        next_downgrade_to_free.ge(base["snapshot_date"]) & 
        next_downgrade_to_free.lt(horizon)
    )
    
    # 3. Determine inactivity in next 30 days
    completed_orders = orders[orders["_is_completed"] == 1][["customer_id", order_date]]
    successful_payments = payments[payments["_is_success"] == 1][["customer_id", payment_date]]
    activity_events = pd.concat([
        usage[["customer_id", usage_date]].rename(columns={usage_date: "activity_date"}),
        completed_orders.rename(columns={order_date: "activity_date"}),
        successful_payments.rename(columns={payment_date: "activity_date"}),
    ], ignore_index=True)
    
    next_activity = next_event_after_snapshots(base, activity_events, "activity_date")
    next_activity = pd.to_datetime(next_activity)
    inactive_in_window = next_activity.isna() | next_activity.ge(horizon)
    
    # 4. Compute Rule Flags
    # Rule 1: Closed account in prediction window
    closed_df = customers[["customer_id", "closed_date"]].drop_duplicates("customer_id")
    df_labels = base.merge(closed_df, on="customer_id", how="left")
    rule1_closed = (
        df_labels["closed_date"].ge(df_labels["snapshot_date"]) & 
        df_labels["closed_date"].lt(horizon)
    ).astype("int8")
    
    # Rule 2: Downgrade to Free + Inactive
    rule2_downgrade_to_free_inactive = (
        (df_labels["tier_at_snapshot"].notna()) &
        (df_labels["tier_at_snapshot"] != "Free") &
        (downgrade_to_free_in_next_30d == True) &
        (inactive_in_window == True)
    ).astype("int8")
    
    # Rule 3: Already Free + Inactive
    rule3_free_at_snapshot_inactive = (
        (df_labels["tier_at_snapshot"] == "Free") &
        (downgrade_to_free_in_next_30d == False) &
        (inactive_in_window == True)
    ).astype("int8")
    
    # Churn next 30d
    churn_next_30d = (
        (rule1_closed == 1) |
        (rule2_downgrade_to_free_inactive == 1) |
        (rule3_free_at_snapshot_inactive == 1)
    ).astype("int8")
    
    # Determine reason
    reasons = []
    for r1, r2, r3 in zip(rule1_closed, rule2_downgrade_to_free_inactive, rule3_free_at_snapshot_inactive):
        sub_reasons = []
        if r1 == 1:
            sub_reasons.append("Closed")
        if r2 == 1:
            sub_reasons.append("DowngradeToFree_Inactive")
        if r3 == 1:
            sub_reasons.append("FreeTier_Inactive")
        if not sub_reasons:
            reasons.append("Active")
        else:
            reasons.append("+".join(sub_reasons))
            
    res_df = pd.DataFrame({
        "customer_id": base["customer_id"],
        "snapshot_date": base["snapshot_date"],
        "churn_next_30d": churn_next_30d,
        "rule1_closed": rule1_closed,
        "rule2_downgrade_to_free_inactive": rule2_downgrade_to_free_inactive,
        "rule3_free_at_snapshot_inactive": rule3_free_at_snapshot_inactive,
        "churn_reason": reasons,
        "tier_at_snapshot": base["tier_at_snapshot"].fillna("Unknown")
    })
    return res_df


## 3. Tạo Tập Dữ Liệu & Audit Nhãn Churn v2 (Rebuild from scratch)

Chạy trích xuất đặc trưng lags, rolling, trend, recency từ các bảng sự kiện thô và ghép nối nhãn v2.

In [ ]:
logger.info("Building temporal feature engineering from raw Silver tables...")

# 1. Base temporal features (Monthly aggregation)
base = build_temporal_base_features()
# Let base["customer_id"] keep its natural type (int32) during feature extraction to match other Silver tables
behavioral = [c for c in base.columns if c not in {"customer_id", "snapshot_date"}]

# 2. Lag features
df_lag = build_lag_features(base, behavioral)
lag_cols = [c for c in df_lag.columns if "_lag_" in c]

# 3. Rolling features
df_roll = build_rolling_features(base, behavioral)
roll_cols = [c for c in df_roll.columns if "_rolling_" in c]

# 4. Trend features
df_trend = build_trend_features(base, behavioral)
trend_cols = [c for c in df_trend.columns if "_change_" in c or "_slope_" in c]

# 5. Recency features
df_rec = build_recency_features(base)
rec_cols = [c for c in df_rec.columns if "days_since_last_" in c]

# 6. Recompute churn labels using v2 rules
res_labels = build_churn_labels_v2(base)

# 7. Merge all features and labels
logger.info("Merging features and labels into final dataset...")
merged = base.merge(df_lag[["customer_id", "snapshot_date"] + lag_cols], on=["customer_id", "snapshot_date"], how="left")
merged = merged.merge(df_roll, on=["customer_id", "snapshot_date"], how="left")
merged = merged.merge(df_trend, on=["customer_id", "snapshot_date"], how="left")
merged = merged.merge(df_rec, on=["customer_id", "snapshot_date"], how="left")

# Cast merged["customer_id"] to int64 to align with res_labels["customer_id"] (which is int64)
merged["customer_id"] = merged["customer_id"].astype("int64")
df_v2 = merged.merge(res_labels, on=["customer_id", "snapshot_date"], how="inner")

# 8. Chronological Preprocessing (Median Imputation)
logger.info("Applying chronological preprocessing (median imputation strictly fit on Train)...")
train_end = pd.Timestamp("2025-08-01")
meta_cols = {
    "customer_id", "snapshot_date", "churn_next_30d",
    "rule1_closed", "rule2_downgrade_to_free_inactive", "rule3_free_at_snapshot_inactive",
    "churn_reason", "tier_at_snapshot", "closed_date", "label_complete"
}
feature_cols = [c for c in df_v2.columns if c not in meta_cols]

# Compute medians strictly on Train split (<= 2025-08-01)
train_mask = pd.to_datetime(df_v2["snapshot_date"]) <= train_end
train_medians = df_v2.loc[train_mask, feature_cols].median()

# Impute the entire merged dataset
df_v2[feature_cols] = df_v2[feature_cols].fillna(train_medians)
df_v2[feature_cols] = df_v2[feature_cols].fillna(0.0) # Fallback for any all-NaN columns on Train

# Save output parquet
output_path = config.OUTPUT_DIR / "churn_temporal_dataset_v2.parquet"
df_v2.to_parquet(output_path, index=False)
logger.info(f"Preprocessed temporal dataset v2 successfully generated at {output_path}. Shape: {df_v2.shape}")

# Save output CSV
output_csv = config.OUTPUT_DIR / "churn_temporal_dataset_v2.csv"
df_v2.to_csv(output_csv, index=False)
logger.info(f"Preprocessed temporal dataset v2 successfully saved to CSV at {output_csv}")

# ==========================================
# 9. LABEL AUDIT
# ==========================================
logger.info("Generating Label Audit Report...")
total_rows = len(res_labels)
total_churn = int(res_labels["churn_next_30d"].sum())
non_churn = total_rows - total_churn
churn_rate = total_churn / total_rows if total_rows > 0 else 0.0

r1 = res_labels["rule1_closed"] == 1
r2 = res_labels["rule2_downgrade_to_free_inactive"] == 1
r3 = res_labels["rule3_free_at_snapshot_inactive"] == 1

r1_count = int(r1.sum())
r2_count = int(r2.sum())
r3_count = int(r3.sum())

r1_only = int((r1 & ~r2 & ~r3).sum())
r2_only = int((~r1 & r2 & ~r3).sum())
r3_only = int((~r1 & ~r2 & r3).sum())

r1_r2 = int((r1 & r2 & ~r3).sum())
r1_r3 = int((r1 & ~r2 & r3).sum())
r2_r3 = int((~r1 & r2 & r3).sum())
all_3 = int((r1 & r2 & r3).sum())

audit_data = {
    "metric": [
        "Total customer-snapshots",
        "Total churn",
        "Non-churn",
        "Churn rate",
        "Rule 1 count",
        "Rule 2 count",
        "Rule 3 count",
        "Rule1 only",
        "Rule2 only",
        "Rule3 only",
        "Rule1 + Rule2",
        "Rule1 + Rule3",
        "Rule2 + Rule3",
        "All 3"
    ],
    "value": [
        total_rows,
        total_churn,
        non_churn,
        churn_rate,
        r1_count,
        r2_count,
        r3_count,
        r1_only,
        r2_only,
        r3_only,
        r1_r2,
        r1_r3,
        r2_r3,
        all_3
    ]
}
audit_df = pd.DataFrame(audit_data)
audit_csv_path = config.OUTPUT_DIR / "churn_rule_v2_audit.csv"
audit_df.to_csv(audit_csv_path, index=False)
logger.info(f"Audit report saved to {audit_csv_path}")

print("\n============================================================")
print("NEW CHURN RULE AUDIT SUMMARY")
print("============================================================")
for idx, row in audit_df.iterrows():
    val = row['value']
    if isinstance(val, float):
        print(f"{row['metric'] + ':':<35} {val:.6%}")
    else:
        print(f"{row['metric'] + ':':<35} {val:,}")
print("============================================================")

# ==========================================
# 10. LABEL DISTRIBUTION BY SNAPSHOT
# ==========================================
logger.info("Generating Label Distribution by Snapshot...")
snap_grouped = res_labels.groupby("snapshot_date").agg(
    customers=("churn_next_30d", "count"),
    rule1_count=("rule1_closed", "sum"),
    rule2_count=("rule2_downgrade_to_free_inactive", "sum"),
    rule3_count=("rule3_free_at_snapshot_inactive", "sum"),
    churn_count=("churn_next_30d", "sum")
).reset_index()
snap_grouped["churn_rate"] = snap_grouped["churn_count"] / snap_grouped["customers"]

snap_csv_path = config.OUTPUT_DIR / "churn_rule_v2_by_snapshot.csv"
snap_grouped.to_csv(snap_csv_path, index=False)
logger.info(f"Distribution by snapshot saved to {snap_csv_path}")

print("\n============================================================")
print("LABEL DISTRIBUTION BY SNAPSHOT DATE")
print("============================================================")
print(snap_grouped.to_string(index=False))
print("============================================================")


## 4. Huấn Luyện, Hiệu Chuẩn Platt và Tối Ưu Hóa Ngưỡng Mô Hình v2

Chia Train/Val/Test, huấn luyện LightGBM và Platt calibrator.

In [ ]:
def get_max_data_date() -> pd.Timestamp:
    orders = load_silver_table("churn_orders")
    order_date = next(c for c in ("order_date", "created_at") if c in orders.columns)
    max_order = pd.to_datetime(orders[order_date]).max()
    
    payments = load_silver_table("churn_payments")
    pay_date = next(c for c in ("payment_date", "created_at") if c in payments.columns)
    max_pay = pd.to_datetime(payments[pay_date]).max()
    
    max_date = max(max_order, max_pay)
    if getattr(max_date, "tz", None) is not None:
        max_date = max_date.tz_localize(None)
    return max_date

# Load dataset v2
data_path = config.OUTPUT_DIR / "churn_temporal_dataset_v2.parquet"
df_v2 = pd.read_parquet(data_path)
df_v2["snapshot_date"] = pd.to_datetime(df_v2["snapshot_date"])

max_data_date = get_max_data_date()
logger.info(f"Latest database event date: {max_data_date.date()}")

# Split ranges
train_start = pd.Timestamp("2024-09-01")
train_end = pd.Timestamp("2025-08-01")
val_start = pd.Timestamp("2025-09-01")
val_end = pd.Timestamp("2026-02-01")
test_start = pd.Timestamp("2026-03-01")
test_end = pd.Timestamp("2026-06-01")

# Maturity checks
for name, end_date in [("Train", train_end), ("Validation", val_end), ("Test", test_end)]:
    maturity_date = end_date + pd.Timedelta(days=30)
    if maturity_date > max_data_date:
        raise ValueError(
            f"FAIL-SAFE: Cannot train model. Labels for {name} end snapshot {end_date.date()} "
            f"are immature! Require data up to {maturity_date.date()} but database only has events up to {max_data_date.date()}."
        )

train_mask = (df_v2["snapshot_date"] >= train_start) & (df_v2["snapshot_date"] <= train_end)
val_mask = (df_v2["snapshot_date"] >= val_start) & (df_v2["snapshot_date"] <= val_end)
test_mask = (df_v2["snapshot_date"] >= test_start) & (df_v2["snapshot_date"] <= test_end)

train_rows = int(train_mask.sum())
val_rows = int(val_mask.sum())
test_rows = int(test_mask.sum())

train_churn = int(df_v2.loc[train_mask, "churn_next_30d"].sum())
val_churn = int(df_v2.loc[val_mask, "churn_next_30d"].sum())
test_churn = int(df_v2.loc[test_mask, "churn_next_30d"].sum())

train_rate = train_churn / train_rows if train_rows > 0 else 0.0
val_rate = val_churn / val_rows if val_rows > 0 else 0.0
test_rate = test_churn / test_rows if test_rows > 0 else 0.0

print("\n============================================================")
print("CHRONOLOGICAL SPLITS DIAGNOSTICS")
print("============================================================")
print(f"Train Window:       {train_start.date()} to {train_end.date()}")
print(f"Train Rows:         {train_rows:,}")
print(f"Train Churn Count:  {train_churn:,}")
print(f"Train Churn Rate:   {train_rate:.6%}")
print("------------------------------------------------------------")
print(f"Val Window:         {val_start.date()} to {val_end.date()}")
print(f"Val Rows:           {val_rows:,}")
print(f"Val Churn Count:    {val_churn:,}")
print(f"Val Churn Rate:     {val_rate:.6%}")
print("------------------------------------------------------------")
print(f"Test Window:        {test_start.date()} to {test_end.date()}")
print(f"Test Rows:          {test_rows:,}")
print(f"Test Churn Count:   {test_churn:,}")
print(f"Test Churn Rate:    {test_rate:.6%}")
print("============================================================")

# Identify features
meta_cols = {
    "customer_id", "snapshot_date", "churn_next_30d",
    "rule1_closed", "rule2_downgrade_to_free_inactive", "rule3_free_at_snapshot_inactive",
    "churn_reason", "tier_at_snapshot", "closed_date", "label_complete"
}
selected_features = [c for c in df_v2.columns if c not in meta_cols and not c.startswith("future_") and not c.endswith("_future")]
logger.info(f"Number of training features: {len(selected_features)}")

X_train = df_v2.loc[train_mask, selected_features].copy()
y_train = df_v2.loc[train_mask, "churn_next_30d"].to_numpy()

X_val = df_v2.loc[val_mask, selected_features].copy()
y_val = df_v2.loc[val_mask, "churn_next_30d"].to_numpy()

X_test = df_v2.loc[test_mask, selected_features].copy()
y_test = df_v2.loc[test_mask, "churn_next_30d"].to_numpy()

# Preprocessing
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

# Fit LightGBM
seed = 42
np.random.seed(seed)
sw = (len(y_train) - y_train.sum()) / y_train.sum()

hyperparams = {
    "num_leaves": 15,
    "max_depth": 4,
    "learning_rate": 0.01,
    "n_estimators": 400,
    "min_child_samples": 50,
    "feature_fraction": 0.7,
    "bagging_fraction": 1.0,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0
}

logger.info("Fitting LightGBM classifier on Train split...")
clf = lgb.LGBMClassifier(
    random_state=seed,
    scale_pos_weight=sw,
    verbose=-1,
    **hyperparams
)
clf.fit(X_train_imp, y_train)

# Fit Platt Scaling Calibration
logger.info("Calibrating model on Validation split probabilities...")
val_probs_raw = clf.predict_proba(X_val_imp)[:, 1]
calibrator = fit_platt_calibration(val_probs_raw, y_val)

# Optimize threshold
val_probs_cal = calibrator.predict_proba(val_probs_raw.reshape(-1, 1))[:, 1]
best_f1 = -1.0
best_thresh = config.DEFAULT_THRESHOLD
for th in np.linspace(0.01, 0.99, 99):
    preds = (val_probs_cal >= th).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = float(th)

logger.info(f"Optimal validation threshold selected: {best_thresh:.2f} (Val F1: {best_f1:.4%})")

# Evaluate
train_probs_raw = clf.predict_proba(X_train_imp)[:, 1]
train_probs_cal = calibrator.predict_proba(train_probs_raw.reshape(-1, 1))[:, 1]

test_probs_raw = clf.predict_proba(X_test_imp)[:, 1]
test_probs_cal = calibrator.predict_proba(test_probs_raw.reshape(-1, 1))[:, 1]

train_metrics = evaluate_predictions(y_train, train_probs_cal, best_thresh)
val_metrics = evaluate_predictions(y_val, val_probs_cal, best_thresh)
test_metrics = evaluate_predictions(y_test, test_probs_cal, best_thresh)

# Save bundle v2
bundle = {
    "model": clf,
    "calibrator": calibrator,
    "selected_features": selected_features,
    "imputer": imputer,
    "threshold": best_thresh,
    "business_rule_version": "v2",
    "label_definition": "v2 (Rule 1: Closed, Rule 2: Downgrade to Free + Inactive, Rule 3: Already Free + Inactive)",
    "training_start": str(train_start.date()),
    "training_end": str(train_end.date()),
    "feature_count": len(selected_features),
    "random_seed": seed,
    "created_at": datetime.datetime.now().isoformat()
}

artifact_path = config.ARTIFACTS_DIR / "temporal_churn_model_v2.joblib"
joblib.dump(bundle, artifact_path)
logger.info(f"Model v2 bundle successfully saved to {artifact_path}")

print("\n============================================================")
print("FINAL PRODUCTION AUDIT REPORT: MODEL TRAINING (V2)")
print("============================================================")
print(f"Random Seed:          {seed}")
print(f"Features Count:       {len(selected_features)}")
print(f"Calibrated Threshold: {best_thresh:.2f}")
print("------------------------------------------------------------")
print("VALIDATION METRICS:")
print(f"Validation PR-AUC:    {val_metrics['PR_AUC']:.6f}")
print(f"Validation ROC-AUC:   {val_metrics['ROC_AUC']:.6f}")
print(f"Validation Precision: {val_metrics['precision']:.4%}")
print(f"Validation Recall:    {val_metrics['recall']:.4%}")
print(f"Validation F1:        {val_metrics['F1']:.4%}")
print(f"Validation Brier Src: {val_metrics['Brier']:.6f}")
print("------------------------------------------------------------")
print("CLEAN TEST METRICS:")
print(f"Test PR-AUC:          {test_metrics['PR_AUC']:.6f}")
print(f"Test ROC-AUC:         {test_metrics['ROC_AUC']:.6f}")
print(f"Test Precision:       {test_metrics['precision']:.4%}")
print(f"Test Recall:          {test_metrics['recall']:.4%}")
print(f"Test F1:              {test_metrics['F1']:.4%}")
print(f"Test Brier Score:     {test_metrics['Brier']:.6f}")
conf_mtx = [[test_metrics['TN'], test_metrics['FP']], [test_metrics['FN'], test_metrics['TP']]]
print(f"Test Confusion Mtx:\n{conf_mtx}")
print("============================================================")


## 5. Kiểm Định & Đánh Giá Mô Hình v2 đã lưu

Đánh giá các chỉ số hiệu năng của mô hình v2 trên tập Test sạch.

In [ ]:
def evaluate_bundle(bundle_path: Path, dataset_path: Path) -> dict:
    bundle = joblib.load(bundle_path)
    df = pd.read_parquet(dataset_path)
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
    
    test_start = pd.Timestamp("2026-03-01")
    test_end = pd.Timestamp("2026-06-01")
    test_df = df[(df["snapshot_date"] >= test_start) & (df["snapshot_date"] <= test_end)]
    
    target_col = "churn_next_30d"
    total_rows = len(df)
    churn_count = int(df[target_col].sum())
    churn_rate = churn_count / total_rows if total_rows > 0 else 0.0
    
    selected_features = bundle["selected_features"]
    imputer = bundle["imputer"]
    clf = bundle["model"]
    calibrator = bundle["calibrator"]
    threshold = bundle["threshold"]
    
    X_test = test_df[selected_features].copy()
    y_test = test_df[target_col].to_numpy()
    
    X_test_imp = imputer.transform(X_test)
    raw_probs = clf.predict_proba(X_test_imp)[:, 1]
    cal_probs = calibrator.predict_proba(raw_probs.reshape(-1, 1))[:, 1]
    
    metrics = evaluate_predictions(y_test, cal_probs, threshold)
    
    return {
        "total_rows": total_rows,
        "churn_count": churn_count,
        "churn_rate": churn_rate,
        "PR_AUC": metrics["PR_AUC"],
        "ROC_AUC": metrics["ROC_AUC"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "F1": metrics["F1"]
    }

# Evaluating Model v2
bundle_v2_path = config.ARTIFACTS_DIR / "temporal_churn_model_v2.joblib"
dataset_v2_path = config.OUTPUT_DIR / "churn_temporal_dataset_v2.parquet"

res_v2 = evaluate_bundle(bundle_v2_path, dataset_v2_path)

# Create Metrics DataFrame for Model v2
metrics_data = {
    "Metric": ["Total Rows", "Churn Count", "Churn Rate", "PR-AUC", "ROC-AUC", "Precision", "Recall", "F1"],
    "Value": [
        f"{res_v2['total_rows']:,}", f"{res_v2['churn_count']:,}", f"{res_v2['churn_rate']:.6%}",
        f"{res_v2['PR_AUC']:.6f}", f"{res_v2['ROC_AUC']:.6f}", f"{res_v2['precision']:.4%}",
        f"{res_v2['recall']:.4%}", f"{res_v2['F1']:.4%}"
    ]
}
metrics_df = pd.DataFrame(metrics_data)

metrics_csv_path = config.OUTPUT_DIR / "churn_rule_v2_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
logger.info(f"Metrics report saved to {metrics_csv_path}")

print("\n============================================================")
print("PRODUCTION MODEL V2 PERFORMANCE METRICS")
print("============================================================")
print(metrics_df.to_string(index=False))
print("============================================================")

# Verify saved model bundle v2 loading & prediction range
logger.info("Verifying saved model bundle v2...")
bundle_v2 = joblib.load(bundle_v2_path)
df_v2 = pd.read_parquet(dataset_v2_path)
sample_df = df_v2.sample(n=10, random_state=42)

features = bundle_v2["selected_features"]
imputer = bundle_v2["imputer"]
model = bundle_v2["model"]
calibrator = bundle_v2["calibrator"]
threshold = bundle_v2["threshold"]

X_sample = sample_df[features].copy()
X_sample_imp = imputer.transform(X_sample)

raw_sample_probs = model.predict_proba(X_sample_imp)[:, 1]
cal_sample_probs = calibrator.predict_proba(raw_sample_probs.reshape(-1, 1))[:, 1]
sample_preds = (cal_sample_probs >= threshold).astype(int)

# Assertions
assert len(features) == len(bundle_v2["selected_features"]), "Feature count mismatch!"
assert np.all(cal_sample_probs >= 0.0) and np.all(cal_sample_probs <= 1.0), "Probabilities out of bounds!"
assert np.all(np.isin(sample_preds, [0, 1])), "Predictions are not binary!"

print("\n============================================================")
print("SAVED MODEL V2 VERIFICATION")
print("============================================================")
print("Model Load:           PASS")
print(f"Feature Compatibility: PASS (Checked {len(features)} features)")
print(f"Probabilities range:   PASS (Min: {cal_sample_probs.min():.4f}, Max: {cal_sample_probs.max():.4f})")
print(f"Prediction output:    PASS ({sample_preds.tolist()})")
print("============================================================")
